In [1]:
import pandas as pd
import numpy as np

In [2]:
df_toy = pd.DataFrame({"x": [10, 12, np.nan, 14, 100, 15, np.nan, 13]})
train, test = df_toy.iloc[:5].copy(), df_toy.iloc[5:].copy()

In [6]:
# This is wrong because we don't want to use test data for complete the missing value
wrong_mean = df_toy["x"].mean()
print("Mean from all data:", wrong_mean)

Mean from all data: 27.333333333333332


In [5]:
# This is right because we only used train data for complete the missing value
train_mean = train["x"].mean()
print("Mean from train:", train_mean)

Mean from train: 34.0


In [7]:
train_wrong = train["x"].fillna(wrong_mean)
train_correct = train["x"].fillna(train_mean)

In [8]:
print("\nWrong:\n", train_wrong.values)
print("Right:\n", train_correct.values)


Wrong:
 [ 10.          12.          27.33333333  14.         100.        ]
Right:
 [ 10.  12.  34.  14. 100.]


In [9]:
test_wrong = test["x"].fillna(wrong_mean)
test_correct = test["x"].fillna(train_mean)

print("Test - Wrong approach:\n", test_wrong.values)
print("Test - Right approach:\n", test_correct.values)

Test - Wrong approach:
 [15.         27.33333333 13.        ]
Test - Right approach:
 [15. 34. 13.]


In [11]:
import duckdb
duckdb.sql("CREATE VIEW duolingo_flagship AS SELECT * FROM read_csv_auto('../../data/duolingo_flagship_v3.csv')")

duckdb.sql("""
SELECT pos, COUNT(*) as total, 
       SUM(CASE WHEN grammar_tags IS NULL THEN 1 ELSE 0 END) as missing,
       ROUND(100.0 * SUM(CASE WHEN grammar_tags IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) as missing_pct
FROM duolingo_flagship
GROUP BY pos
ORDER BY missing DESC
""").show()

┌─────────┬───────┬─────────┬─────────────┐
│   pos   │ total │ missing │ missing_pct │
│ varchar │ int64 │ int128  │   double    │
├─────────┼───────┼─────────┼─────────────┤
│ adv     │   595 │     502 │        84.4 │
│ pr      │   491 │     419 │        85.3 │
│ cnjcoo  │   195 │     195 │       100.0 │
│ adj     │   855 │     174 │        20.4 │
│ ij      │   158 │     158 │       100.0 │
│ cnjadv  │    72 │      72 │       100.0 │
│ cnjsub  │    38 │      38 │       100.0 │
│ n       │  7009 │      31 │         0.4 │
│ preadv  │    18 │      18 │       100.0 │
│ apos    │    16 │      16 │       100.0 │
│  ·      │     · │       · │          ·  │
│  ·      │     · │       · │          ·  │
│  ·      │     · │       · │          ·  │
│ prn     │  1331 │       0 │         0.0 │
│ pprep   │     2 │       0 │         0.0 │
│ vbser   │   741 │       0 │         0.0 │
│ np      │     1 │       0 │         0.0 │
│ predet  │     1 │       0 │         0.0 │
│ vbdo    │    35 │       0 │   

In [13]:
df = duckdb.sql("SELECT * FROM duolingo_flagship").df()
df["grammar_tags"] = df["grammar_tags"].fillna("no_gram")

In [21]:
df["grammar_tags"].isna().sum()

np.int64(0)

In [22]:
df["grammar_tags"].value_counts().head()

grammar_tags
sg           2500
no_gram      1714
pri;p3;sg    1255
m;sg         1250
f;sg         1033
Name: count, dtype: int64

In [23]:
df.to_csv("../../data/duolingo_flagship_v4.csv", index=False)